# **Machine Learning Assignment**
## Email Classification: Spam / Ham  
### Manual Implementation of a Multinomial Naive Bayes Model

This notebook presents a complete supervised machine learning flow for classifying emails as either spam or ham.  
The task is based on Natural Language Processing (NLP), where raw text is transformed into numerical features and then classified using a manually implemented learning algorithm.

## Students
**Oriya Peretz & Omri Kuperberg**

## Assignment Details

- Assignment type: **Text Analysis / NLP**
- Learning type: **Classification**
- Problem type: **Binary Classification**
- Main class: **Spam**
- Dataset name: **Spam vs Ham Emails**
- Dataset URL: **https://www.kaggle.com/datasets/yashpaloswal/spamham-email-classification-nlp**
- Implemented learning algorithm: **Multinomial Naive Bayes**


# 1. Learning Problem and Dataset Explanation

The goal of this assignment is to build a model that can classify an email as either spam or ham based on its textual content.

Each sample in the dataset represents one email.  
The input feature is the email text, and the target label indicates whether the email is spam.

The target column is:
- `1` — Spam email
- `0` — Ham email

This is a supervised learning problem because the dataset contains labeled examples.  
It is also a binary classification problem because there are only two possible classes.

Since the main objective is to detect spam emails, the positive and main class in this task is `Spam`.  
Therefore, the main quality metric used in this notebook is the F1-score for the spam class.

# 2. Import Libraries

The following libraries are used in this notebook:

- `pandas` for loading and displaying the dataset.
- `numpy` for numerical calculations.
- `re` for text cleaning using regular expressions.
- `math` for logarithmic probability calculations.
- `sklearn` utilities for splitting, vectorization, cross validation splitting, and evaluation metrics.

The model itself is implemented manually.

In [ ]:
import pandas as pd
import numpy as np
import re
import math

from sklearn.model_selection import train_test_split, KFold
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.metrics import f1_score, precision_score, recall_score, accuracy_score, confusion_matrix

# 3. Load the Dataset

The dataset file should be uploaded to the Colab environment under the name `emails.csv`.

If the file has a different name, the filename in the next cell should be updated accordingly.

In [ ]:
df = pd.read_csv("emails.csv")

df.head()

,Text,Spam
0,Subject: naturally irresistible your corporate...,1
1,Subject: the stock trading gunslinger fanny i...,1
2,Subject: unbelievable new homes made easy im ...,1
3,Subject: 4 color printing special request add...,1
4,"Subject: do not have money , get software cds ...",1


# 4. Initial Data Inspection

Before training a model, it is important to inspect the dataset.

In this step we check:
- The number of rows and columns.
- The data types of the columns.
- Whether there are missing values.
- The distribution of the target classes.

In [ ]:
print(df.info())
print()

print("Missing values:")
print(df.isnull().sum())
print()

print("Class distribution:")
print(df["Spam"].value_counts())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5728 entries, 0 to 5727
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   Text    5728 non-null   object
 1   Spam    5728 non-null   int64 
dtypes: int64(1), object(1)
memory usage: 89.6+ KB
None

Missing values:
Text    0
Spam    0
dtype: int64

Class distribution:
Spam
0    4360
1    1368
Name: count, dtype: int64


The dataset contains two main columns:

- `Text`: the content of the email.
- `Spam`: the target label.

The model will learn patterns in the email text in order to predict whether the email is spam or ham.

# 5. Text Preprocessing

Raw text usually contains noise such as uppercase letters, punctuation marks, numbers, and extra spaces.  
Cleaning the text helps create a more consistent representation before feature extraction.

The preprocessing performed here includes:
- Converting all text to lowercase.
- Removing the word `subject`, which appears frequently in email datasets.
- Removing characters that are not English letters.
- Removing extra spaces.

This step helps reduce unnecessary variations in the text.

In [ ]:
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"subject:", " ", text)
    text = re.sub(r"[^a-z\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

df["clean_text"] = df["Text"].apply(clean_text)

df[["Text", "clean_text", "Spam"]].head()

,Text,clean_text,Spam
0,Subject: naturally irresistible your corporate...,naturally irresistible your corporate identity...,1
1,Subject: the stock trading gunslinger fanny i...,the stock trading gunslinger fanny is merrill ...,1
2,Subject: unbelievable new homes made easy im ...,unbelievable new homes made easy im wanting to...,1
3,Subject: 4 color printing special request add...,color printing special request additional info...,1
4,"Subject: do not have money , get software cds ...",do not have money get software cds from here s...,1


# 6. Train/Test Split

The dataset is split into a training set and a test set.

- The training set is used to fit the model.
- The test set is used only at the end to evaluate the final model on unseen data.

We use an 80/20 split.  
The `stratify` parameter keeps the class distribution similar in both the training and test sets.

In [ ]:
train_df, test_df = train_test_split(
    df,
    test_size=0.2,
    random_state=42,
    stratify=df["Spam"]
)

print("Train size:", len(train_df))
print("Test size:", len(test_df))

Train size: 4582
Test size: 1146


## 6.1 First Five Rows of the Training Set

In [ ]:
train_df.head()

,Text,Spam,clean_text
1428,"Subject: re : a personal favor anurag , i sh...",0,re a personal favor anurag i shall talk about ...
4688,Subject: site license for power world gentlem...,0,site license for power world gentlemen i recom...
1364,Subject: would you like a $ 250 gas card ? do...,1,would you like a gas card don t let the curren...
1671,Subject: erequest password your erequest ' s ...,0,erequest password your erequest s password is ...
2648,Subject: re : change - video to teleconference...,0,re change video to teleconferences enron chris...


## 6.2 First Five Rows of the Test Set

In [ ]:
test_df.head()

,Text,Spam,clean_text
897,"Subject: good day friend dear friend , my na...",1,good day friend dear friend my name is salim i...
511,Subject: wallstreet pulse good day to all bro...,1,wallstreet pulse good day to all broker s day ...
5274,Subject: sap time sheets on the o : \ research...,0,sap time sheets on the o research common drive...
3524,Subject: holiday gift thank you so much for y...,0,holiday gift thank you so much for your though...
2,Subject: unbelievable new homes made easy im ...,1,unbelievable new homes made easy im wanting to...


# 7. Feature Engineering

Machine learning models cannot directly understand raw text.  
Therefore, the text must be converted into numerical features.

In this notebook, two feature extraction methods are examined:

1. **Bag of Words** using `CountVectorizer`  
   This method represents each email by counting how many times each word appears.

2. **TF-IDF** using `TfidfVectorizer`  
   This method gives higher weight to words that are important in a specific document but not too common across the entire dataset.

Both methods are useful for text classification.  
The model itself remains manually implemented in both cases.

## 7.1 Examples Before and After Cleaning

The following examples demonstrate how the original email text looks before and after preprocessing.

In [ ]:
train_df[["Text", "clean_text", "Spam"]].head(3)

,Text,clean_text,Spam
1428,"Subject: re : a personal favor anurag , i sh...",re a personal favor anurag i shall talk about ...,0
4688,Subject: site license for power world gentlem...,site license for power world gentlemen i recom...,0
1364,Subject: would you like a $ 250 gas card ? do...,would you like a gas card don t let the curren...,1


## 7.2 Bag of Words Example

Bag of Words converts text into a matrix of word counts.  
Each column represents a word, and each row represents an email.

In [ ]:
bow_demo_vectorizer = CountVectorizer(max_features=20)
bow_demo_matrix = bow_demo_vectorizer.fit_transform(train_df["clean_text"].head(3))

bow_demo_df = pd.DataFrame(
    bow_demo_matrix.toarray(),
    columns=bow_demo_vectorizer.get_feature_names_out()
)

bow_demo_df

,and,ect,enron,flows,for,have,in,is,license,load,of,on,option,power,site,the,this,to,vince,you
0,2,1,2,0,4,3,3,3,0,0,5,2,0,0,0,4,1,12,2,6
1,7,5,5,4,10,1,2,0,5,7,4,2,4,5,5,9,6,8,4,0
2,0,0,0,0,0,0,1,1,0,0,1,0,0,0,0,1,1,2,0,2


## 7.3 TF-IDF Example

TF-IDF also creates numerical features, but instead of using only raw word counts, it assigns a weight to each word.

A word receives a high TF-IDF value when:
- It appears frequently in a specific email.
- It does not appear too frequently across many emails.

This can help reduce the influence of very common words.

In [ ]:
tfidf_demo_vectorizer = TfidfVectorizer(max_features=20)
tfidf_demo_matrix = tfidf_demo_vectorizer.fit_transform(train_df["clean_text"].head(3))

tfidf_demo_df = pd.DataFrame(
    tfidf_demo_matrix.toarray(),
    columns=tfidf_demo_vectorizer.get_feature_names_out()
)

tfidf_demo_df

,and,ect,enron,flows,for,have,in,is,license,load,of,on,option,power,site,the,this,to,vince,you
0,0.139822,0.069911,0.139822,0.000000,0.279643,0.209732,0.162876,0.209732,0.000000,0.000000,0.271460,0.139822,0.000000,0.000000,0.000000,0.217168,0.054292,0.651504,0.139822,0.419465
1,0.283206,0.202290,0.202290,0.212789,0.404580,0.040458,0.062838,0.000000,0.265987,0.372381,0.125677,0.080916,0.212789,0.265987,0.265987,0.282773,0.188515,0.251354,0.161832,0.000000
2,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.247760,0.319036,0.000000,0.000000,0.247760,0.000000,0.000000,0.000000,0.000000,0.247760,0.247760,0.495520,0.000000,0.638072


# 8. Manual Implementation of Multinomial Naive Bayes

Multinomial Naive Bayes is a common algorithm for text classification tasks such as spam detection.

The algorithm is based on Bayes' theorem and estimates the probability of each class given the words that appear in the email.

For each class, the model learns:
- The prior probability of the class.
- The probability of each word appearing in that class.

For prediction, the model calculates a score for each class and selects the class with the highest score.

Laplace smoothing is used through the parameter `alpha`.  
This prevents zero probabilities for words that did not appear in the training examples of a certain class.

The model includes:
- `fit`: learns probabilities from the training data.
- `predict`: predicts labels for new examples.

In [ ]:
class MyMultinomialNaiveBayes:
    def __init__(self, alpha=1.0):
        self.alpha = alpha

    def fit(self, X, y):
        X = X.toarray() if hasattr(X, "toarray") else np.array(X)
        y = np.array(y)

        self.classes = np.unique(y)
        self.class_log_prior = {}
        self.feature_log_prob = {}

        n_samples, n_features = X.shape

        for c in self.classes:
            X_c = X[y == c]

            class_count = X_c.shape[0]
            self.class_log_prior[c] = math.log(class_count / n_samples)

            feature_count = X_c.sum(axis=0)
            total_count = feature_count.sum()

            smoothed_feature_count = feature_count + self.alpha
            smoothed_total_count = total_count + self.alpha * n_features

            self.feature_log_prob[c] = np.log(smoothed_feature_count / smoothed_total_count)

    def predict(self, X):
        X = X.toarray() if hasattr(X, "toarray") else np.array(X)
        predictions = []

        for row in X:
            scores = {}

            for c in self.classes:
                score = self.class_log_prior[c] + np.sum(row * self.feature_log_prob[c])
                scores[c] = score

            predictions.append(max(scores, key=scores.get))

        return np.array(predictions)

# 9. Evaluation Metrics

Since spam is the main class, the main metric is F1-score for the spam class.

F1-score combines:
- **Precision**: Out of all emails predicted as spam, how many were actually spam.
- **Recall**: Out of all actual spam emails, how many were detected.
- **F1-score**: The harmonic mean of precision and recall.

Accuracy is also shown, but F1-score is more important here because the goal is focused on detecting the spam class.

# 10. Training and Evaluation Function

The following function performs the full flow for a single parameter combination:

1. Create the selected vectorizer.
2. Convert the training and test text into numerical features.
3. Train the manually implemented Naive Bayes model.
4. Predict labels on the test set.
5. Calculate evaluation metrics.

In [ ]:
def train_and_evaluate(vectorizer_type="count", max_features=3000, ngram_range=(1, 1), alpha=1.0):
    if vectorizer_type == "count":
        vectorizer = CountVectorizer(max_features=max_features, ngram_range=ngram_range)
    elif vectorizer_type == "tfidf":
        vectorizer = TfidfVectorizer(max_features=max_features, ngram_range=ngram_range)
    else:
        raise ValueError("vectorizer_type must be either 'count' or 'tfidf'")

    X_train = vectorizer.fit_transform(train_df["clean_text"])
    X_test = vectorizer.transform(test_df["clean_text"])

    y_train = train_df["Spam"].values
    y_test = test_df["Spam"].values

    model = MyMultinomialNaiveBayes(alpha=alpha)
    model.fit(X_train, y_train)

    predictions = model.predict(X_test)

    metrics = {
        "accuracy": accuracy_score(y_test, predictions),
        "precision_spam": precision_score(y_test, predictions, pos_label=1, zero_division=0),
        "recall_spam": recall_score(y_test, predictions, pos_label=1, zero_division=0),
        "f1_spam": f1_score(y_test, predictions, pos_label=1, zero_division=0)
    }

    return model, vectorizer, predictions, metrics

# 11. Basic Training Run

First, we train the model using a simple baseline configuration:

- Feature engineering: Bag of Words
- Maximum features: 3000
- N-gram range: unigrams only
- Alpha: 1.0

This provides a first indication that the full training flow works correctly.

In [ ]:
basic_model, basic_vectorizer, basic_predictions, basic_metrics = train_and_evaluate(
    vectorizer_type="count",
    max_features=3000,
    ngram_range=(1, 1),
    alpha=1.0
)

basic_metrics

{'accuracy': 0.9790575916230366,
 'precision_spam': 0.9340277777777778,
 'recall_spam': 0.9817518248175182,
 'f1_spam': 0.9572953736654805}

# 12. First Five Predictions on the Test Set

The following table shows the first five predictions on the test set.  
This allows us to compare the true label with the predicted label.

In [ ]:
basic_results = test_df[["Text", "Spam"]].copy()
basic_results["prediction"] = basic_predictions

basic_results.head(5)

,Text,Spam,prediction
897,"Subject: good day friend dear friend , my na...",1,1
511,Subject: wallstreet pulse good day to all bro...,1,1
5274,Subject: sap time sheets on the o : \ research...,0,0
3524,Subject: holiday gift thank you so much for y...,0,0
2,Subject: unbelievable new homes made easy im ...,1,1


# 13. Focused Grid Search with Cross Validation

In this section, several parameter combinations are compared using cross validation.

The goal is not to test every possible combination, but to compare meaningful alternatives efficiently.

The tested parameters are:
- `vectorizer_type`: CountVectorizer or TF-IDF
- `max_features`: 1000 or 3000
- `ngram_range`: unigrams only
- `alpha`: 1.0

A 3-fold cross validation is used to keep the experiment efficient while still evaluating the model on different validation splits.

In [ ]:
def cross_validate_experiment(train_data, vectorizer_type, max_features, ngram_range, alpha, n_splits=3):
    kfold = KFold(n_splits=n_splits, shuffle=True, random_state=42)
    scores = []

    texts = train_data["clean_text"].values
    labels = train_data["Spam"].values

    for train_index, validation_index in kfold.split(texts):
        fold_train_texts = texts[train_index]
        fold_validation_texts = texts[validation_index]

        fold_train_labels = labels[train_index]
        fold_validation_labels = labels[validation_index]

        if vectorizer_type == "count":
            vectorizer = CountVectorizer(max_features=max_features, ngram_range=ngram_range)
        else:
            vectorizer = TfidfVectorizer(max_features=max_features, ngram_range=ngram_range)

        X_fold_train = vectorizer.fit_transform(fold_train_texts)
        X_fold_validation = vectorizer.transform(fold_validation_texts)

        model = MyMultinomialNaiveBayes(alpha=alpha)
        model.fit(X_fold_train, fold_train_labels)

        fold_predictions = model.predict(X_fold_validation)

        score = f1_score(
            fold_validation_labels,
            fold_predictions,
            pos_label=1,
            zero_division=0
        )

        scores.append(score)

    return np.mean(scores)

In [ ]:
vectorizer_types = ["count", "tfidf"]
max_features_values = [1000, 3000]
ngram_ranges = [(1, 1)]
alpha_values = [1.0]

experiment_results = []

for vectorizer_type in vectorizer_types:
    for max_features in max_features_values:
        for ngram_range in ngram_ranges:
            for alpha in alpha_values:
                mean_f1 = cross_validate_experiment(
                    train_df,
                    vectorizer_type=vectorizer_type,
                    max_features=max_features,
                    ngram_range=ngram_range,
                    alpha=alpha,
                    n_splits=5
                )

                experiment_results.append({
                    "vectorizer_type": vectorizer_type,
                    "max_features": max_features,
                    "ngram_range": str(ngram_range),
                    "alpha": alpha,
                    "mean_f1_spam": mean_f1
                })

experiments_df = pd.DataFrame(experiment_results)
experiments_df = experiments_df.sort_values("mean_f1_spam", ascending=False).reset_index(drop=True)

experiments_df

,vectorizer_type,max_features,ngram_range,alpha,mean_f1_spam
0,count,3000,"(1, 1)",1.0,0.964438
1,tfidf,3000,"(1, 1)",1.0,0.946773
2,count,1000,"(1, 1)",1.0,0.943618
3,tfidf,1000,"(1, 1)",1.0,0.934841


# 14. Best Parameter Combination

The best parameter combination is selected according to the highest average F1-score for the spam class during cross validation.

In [ ]:
best_experiment = experiments_df.iloc[0]

best_experiment

,0
vectorizer_type,count
max_features,3000
ngram_range,"(1, 1)"
alpha,1.0
mean_f1_spam,0.964438


# 15. Final Model Training

After selecting the best parameter combination, the final model is trained again using the full training set.  
Then it is evaluated on the test set, which was not used during cross validation.

In [ ]:
best_vectorizer_type = best_experiment["vectorizer_type"]
best_max_features = int(best_experiment["max_features"])
best_ngram_range = eval(best_experiment["ngram_range"])
best_alpha = float(best_experiment["alpha"])

final_model, final_vectorizer, final_predictions, final_metrics = train_and_evaluate(
    vectorizer_type=best_vectorizer_type,
    max_features=best_max_features,
    ngram_range=best_ngram_range,
    alpha=best_alpha
)

final_metrics

{'accuracy': 0.9790575916230366,
 'precision_spam': 0.9340277777777778,
 'recall_spam': 0.9817518248175182,
 'f1_spam': 0.9572953736654805}

# 16. Final Test Predictions

The following table shows the first five final predictions on the test set.

In [ ]:
final_results = test_df[["Text", "Spam"]].copy()
final_results["prediction"] = final_predictions

final_results.head(5)

,Text,Spam,prediction
897,"Subject: good day friend dear friend , my na...",1,1
511,Subject: wallstreet pulse good day to all bro...,1,1
5274,Subject: sap time sheets on the o : \ research...,0,0
3524,Subject: holiday gift thank you so much for y...,0,0
2,Subject: unbelievable new homes made easy im ...,1,1


# 17. Confusion Matrix

A confusion matrix helps analyze the model predictions in more detail.

It shows:
- True ham predicted as ham
- Ham predicted incorrectly as spam
- Spam predicted incorrectly as ham
- True spam predicted as spam

In [ ]:
cm = confusion_matrix(test_df["Spam"].values, final_predictions)

confusion_df = pd.DataFrame(
    cm,
    index=["Actual Ham (0)", "Actual Spam (1)"],
    columns=["Predicted Ham (0)", "Predicted Spam (1)"]
)

confusion_df

,Predicted Ham (0),Predicted Spam (1)
Actual Ham (0),853,19
Actual Spam (1),5,269


# 18. Explainability: Most Influential Words

To better understand the model, we examine which words are most strongly associated with each class.

For each word, we compare its learned log probability in the spam class with its learned log probability in the ham class.

Words with a higher score are more associated with spam.  
Words with a lower score are more associated with ham.

In [ ]:
feature_names = final_vectorizer.get_feature_names_out()

spam_label = 1
ham_label = 0

word_scores = []

for index, word in enumerate(feature_names):
    spam_score = final_model.feature_log_prob[spam_label][index]
    ham_score = final_model.feature_log_prob[ham_label][index]
    difference = spam_score - ham_score
    word_scores.append((word, difference))

top_spam_words = sorted(word_scores, key=lambda item: item[1], reverse=True)[:20]
top_ham_words = sorted(word_scores, key=lambda item: item[1])[:20]

print("Top words related to Spam:")
for word, score in top_spam_words:
    print(word, score)

print()
print("Top words related to Ham:")
for word, score in top_ham_words:
    print(word, score)

Top words related to Spam:
viagra 6.384425418028769
stationery 6.0159393131616925
projecthoneypot 5.975529774823816
squirrelmail 5.8434700528107495
sex 5.704883889524603
andmanyother 5.590473538346859
paypal 5.590473538346859
oniine 5.461261806866853
qmail 5.461261806866853
dose 5.443870064154984
mailwisconsin 5.443870064154984
photoshop 5.443870064154984
corel 5.408151981552903
rolete 5.408151981552903
privacy 5.371110709872555
ebay 5.35206251490186
professionai 5.35206251490186
wiil 5.35206251490186
xp 5.35206251490186
vnbl 5.332644429044759

Top words related to Ham:
enron -7.905649602749969
ect -7.046917130080582
kaminski -6.866886868441992
vince -6.746179901069559
crenshaw -5.391525631150376
stinson -5.310921598735747
ees -5.224573571687523
kevin -4.860672888000608
hou -4.83663739618512
gibner -4.801716720867489
weather -4.721674013193953
shirley -4.7172784017209155
wharton -4.7017397982931355
derivatives -4.632267425478368
zimin -4.627459723910265
cc -4.585319089281823
tanya -4.4

# 19. Final Summary

This notebook implemented a complete supervised machine learning flow for spam email classification.

The process included:
- Loading and inspecting the dataset.
- Cleaning the raw email text.
- Splitting the data into train and test sets.
- Demonstrating feature engineering with Bag of Words and TF-IDF.
- Manually implementing the Multinomial Naive Bayes algorithm.
- Training the model and predicting labels.
- Evaluating the model using F1-score for the spam class.
- Comparing several parameter combinations using cross validation.
- Training a final model using the best parameter combination.
- Analyzing the results with a confusion matrix and influential words.

The final model was selected according to cross-validation performance and evaluated on a separate test set.